# Layer 1; 32; 3x3; AvgPool

In [ ]:
import os
print(os.environ.get("LD_LIBRARY_PATH"))

/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cublas/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cuda_runtime/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cudnn/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cufft/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cusolver/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cusparse/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/nccl/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cublas/lib:/mnt/c/Users/ASUS/Documents/GitHub/CNN-RNN-LSTM-Scratch/tf-gpu/lib/python3.10/site-packages/nvidia/cuda_runtime/li

## Import Libraries

In [ ]:
import os

import pandas as pd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
tf.test.is_built_with_cuda()

I0000 00:00:1778583496.994650    4174 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


True

## Config

In [ ]:
TRAIN_DIR  = '../../data/intel/seg_train'
VAL_DIR    = '../../data/intel/seg_test'
TEST_DIR   = '../../data/intel/seg_pred'
MODEL_DIR  = '../../data/models/cnn'
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE    = (150, 150)
BATCH_SIZE  = 64
EPOCHS      = 10
NUM_CLASSES = 6
LR          = 1e-4
VAL_SPLIT   = 0.2

## Import Dataset

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

normalization = keras.layers.Rescaling(1.0 / 255.0)

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
)
class_names = train_ds.class_names

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

train_ds = (
    train_ds
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(buffer_size=1000, seed=SEED, reshuffle_each_iteration=True)
    .prefetch(buffer_size=AUTOTUNE)
)

val_ds = (
    val_ds
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)

test_ds = (
    test_ds
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .prefetch(buffer_size=AUTOTUNE)
)

train_count = sum(1 for _ in tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="training",
    seed=SEED, image_size=IMG_SIZE, batch_size=1, label_mode="int"
))
val_count = sum(1 for _ in tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="validation",
    seed=SEED, image_size=IMG_SIZE, batch_size=1, label_mode="int"
))
test_count = sum(1 for _ in tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=1, label_mode="int", shuffle=False
))

print("Kelas      :", {name: i for i, name in enumerate(class_names)})
print(f"Train      : {train_count} gambar")
print(f"Validation : {val_count} gambar")
print(f"Test       : {test_count} gambar")

Found 11230 images belonging to 6 classes.
Found 2804 images belonging to 6 classes.
Found 3000 images belonging to 6 classes.
Kelas      : {'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}
Train      : 11230 gambar
Validation : 2804 gambar
Test       : 3000 gambar


## CNN Model Baseline Architecture

In [ ]:
MODEL_NAME = 'Layer-1-32-3x3-avgpool'
model = keras.Sequential([
    keras.layers.Input(shape=IMG_SIZE + (3,), name='input'),
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='valid', name='conv2s_1'),
    keras.layers.AveragePooling2D((2, 2), name='avgpool_1'),
    keras.layers.Flatten(name='flatten'),
    keras.layers.Dense(128, activation='relu', name='dense_1'),
    keras.layers.Dense(6, activation='softmax', name='output'),
], name=MODEL_NAME)
model.summary()

Model: "Layer-1-32-3x3-avgpool"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2s_1 (Conv2D)               │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ avgpool_1 (AveragePooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 175232)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │    22,429,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,431,494 (85.57 MB)

 Trainable params: 22,431,494 (85.57 MB)

 Non-trainable params: 0 (0.00 B)

## Compilation and Training

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callback = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=f'../../data/models/cnn/{MODEL_NAME}.h5',
        monitor='val_loss', save_best_only=True, verbose=1
    ),
]

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callback,
    verbose=1
)

Epoch 1/3


I0000 00:00:1778584179.694348    5142 service.cc:153] XLA service 0x7064640315a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778584179.694514    5142 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.5.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1778584179.975127    5142 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778584180.581031    5142 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1778584180.671558    5142 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1373__.17
I0000 00:00:1778584192.591304    5142 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 80/351 ━━━━━━━━━━━━━━━━━━━━ 1:36 358ms/step - accuracy: 0.3231 - loss: 5.0501

I0000 00:00:1778584221.718842    5139 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1373__.17


351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step - accuracy: 0.5062 - loss: 2.4281
Epoch 1: val_loss improved from None to 0.84609, saving model to ../../data/models/cnn/baseline_conv2d.h5


351/351 ━━━━━━━━━━━━━━━━━━━━ 206s 543ms/step - accuracy: 0.6117 - loss: 1.2922 - val_accuracy: 0.6947 - val_loss: 0.8461
Epoch 2/3
351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 334ms/step - accuracy: 0.7647 - loss: 0.6588
Epoch 2: val_loss improved from 0.84609 to 0.73226, saving model to ../../data/models/cnn/baseline_conv2d.h5


351/351 ━━━━━━━━━━━━━━━━━━━━ 149s 424ms/step - accuracy: 0.7722 - loss: 0.6444 - val_accuracy: 0.7361 - val_loss: 0.7323
Epoch 3/3
351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - accuracy: 0.8715 - loss: 0.3843
Epoch 3: val_loss did not improve from 0.73226
351/351 ━━━━━━━━━━━━━━━━━━━━ 146s 419ms/step - accuracy: 0.8722 - loss: 0.3771 - val_accuracy: 0.7225 - val_loss: 0.8152
Restoring model weights from the end of the best epoch: 2.


## Evaluation

In [ ]:
y_pred_proba = model.predict(test_ds)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.concatenate([y.numpy() for _, y in test_ds])

macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f"Macro F1 Score: {macro_f1:.4f}")

94/94 ━━━━━━━━━━━━━━━━━━━━ 38s 389ms/step
Macro F1 Score: 0.7361


## Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'],         label='Train Loss')
ax1.plot(history.history['val_loss'],     label='Val Loss')
ax1.set_title(f'{MODEL_NAME} — Loss'); ax1.legend()
ax2.plot(history.history['accuracy'],     label='Train Acc')
ax2.plot(history.history['val_accuracy'], label='Val Acc')
ax2.set_title(f'{MODEL_NAME} — Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, f'{MODEL_NAME}_curve.png'), dpi=100)
plt.show()